# Comparative Analysis: Professor vs AI Model Evaluations

This notebook analyzes and compares student project evaluations from:
- **Professor** (ground truth)
- **Claude Opus** (most capable model)
- **Claude Sonnet** (balanced model)
- **Claude Haiku** (fastest model)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Data Loading & Cleaning

In [ ]:
# Load data
df = pd.read_csv('evaluations.csv')

# Remove empty rows
df = df.dropna(subset=['project_id', 'evaluator'])

# Standardize evaluator names
evaluator_map = {
    'ПРОФЕСОР': 'Professor',
    'AI-OPUS': 'Opus',
    'AI-SONNET': 'Sonnet',
    'AI-HAIKU': 'Haiku'
}
df['evaluator'] = df['evaluator'].map(evaluator_map)

# Convert numeric columns
numeric_cols = ['metap', 'tocnost', 'kviz', 'vkupno']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Total evaluations: {len(df)}")
print(f"Unique projects: {df['project_id'].nunique()}")
print(f"Project types: {df['project_type'].unique().tolist()}")
df.head(8)

In [ ]:
# Create pivot tables for easier comparison
scores_pivot = df.pivot_table(
    index=['project_id', 'project_type'],
    columns='evaluator',
    values='vkupno'
).reset_index()

print(f"Projects with all evaluations: {len(scores_pivot)}")
scores_pivot.head(10)

## 2. Overall Statistics

In [ ]:
# Summary statistics by evaluator
summary = df.groupby('evaluator')['vkupno'].agg(['count', 'mean', 'std', 'min', 'max', 'median']).round(2)
summary.columns = ['Count', 'Mean', 'Std Dev', 'Min', 'Max', 'Median']
summary = summary.reindex(['Professor', 'Opus', 'Sonnet', 'Haiku'])
summary

In [ ]:
# Visual comparison of score distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
order = ['Professor', 'Opus', 'Sonnet', 'Haiku']
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']
sns.boxplot(data=df, x='evaluator', y='vkupno', order=order, palette=colors, ax=axes[0])
axes[0].set_title('Score Distribution by Evaluator', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Evaluator')
axes[0].set_ylabel('Total Score (vkupno)')

# Violin plot with individual points
sns.violinplot(data=df, x='evaluator', y='vkupno', order=order, palette=colors, ax=axes[1], inner='box')
axes[1].set_title('Score Density Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Evaluator')
axes[1].set_ylabel('Total Score (vkupno)')

plt.tight_layout()
plt.savefig('score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Correlation with Professor Scores

In [ ]:
# Calculate correlations
correlations = {}
for model in ['Opus', 'Sonnet', 'Haiku']:
    valid = scores_pivot[['Professor', model]].dropna()
    if len(valid) > 0:
        r, p = stats.pearsonr(valid['Professor'], valid[model])
        correlations[model] = {'Pearson r': round(r, 4), 'p-value': f'{p:.2e}', 'n': len(valid)}

corr_df = pd.DataFrame(correlations).T
corr_df.index.name = 'AI Model'
print("Correlation with Professor Scores:")
corr_df

In [ ]:
# Scatter plots: Professor vs each AI model
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = ['Opus', 'Sonnet', 'Haiku']
colors = ['#3498db', '#9b59b6', '#e74c3c']

for i, (model, color) in enumerate(zip(models, colors)):
    valid = scores_pivot[['Professor', model, 'project_type']].dropna()
    
    # Scatter by project type
    for ptype, marker in [('noAI', 'o'), ('onlyAI', 's'), ('hybrid', '^')]:
        subset = valid[valid['project_type'] == ptype]
        axes[i].scatter(subset['Professor'], subset[model], 
                       alpha=0.7, s=60, marker=marker, label=ptype)
    
    # Perfect agreement line
    axes[i].plot([0, 100], [0, 100], 'k--', alpha=0.5, label='Perfect Agreement')
    
    # Regression line
    z = np.polyfit(valid['Professor'], valid[model], 1)
    p = np.poly1d(z)
    x_line = np.linspace(0, 100, 100)
    axes[i].plot(x_line, p(x_line), color=color, linewidth=2, label=f'Regression')
    
    r = stats.pearsonr(valid['Professor'], valid[model])[0]
    axes[i].set_title(f'Professor vs {model}\n(r = {r:.3f})', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Professor Score')
    axes[i].set_ylabel(f'{model} Score')
    axes[i].set_xlim(0, 100)
    axes[i].set_ylim(0, 100)
    axes[i].legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('correlation_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Scoring Bias Analysis

In [ ]:
# Calculate differences (AI - Professor)
for model in ['Opus', 'Sonnet', 'Haiku']:
    scores_pivot[f'{model}_diff'] = scores_pivot[model] - scores_pivot['Professor']

# Summary of differences
diff_summary = pd.DataFrame({
    'Model': ['Opus', 'Sonnet', 'Haiku'],
    'Mean Diff': [scores_pivot[f'{m}_diff'].mean() for m in ['Opus', 'Sonnet', 'Haiku']],
    'Median Diff': [scores_pivot[f'{m}_diff'].median() for m in ['Opus', 'Sonnet', 'Haiku']],
    'Std Diff': [scores_pivot[f'{m}_diff'].std() for m in ['Opus', 'Sonnet', 'Haiku']],
    'MAE': [scores_pivot[f'{m}_diff'].abs().mean() for m in ['Opus', 'Sonnet', 'Haiku']],
    'RMSE': [np.sqrt((scores_pivot[f'{m}_diff']**2).mean()) for m in ['Opus', 'Sonnet', 'Haiku']]
}).round(2)

print("Scoring Bias Analysis (AI Score - Professor Score):")
print("Positive = AI scores higher, Negative = AI scores lower")
diff_summary.set_index('Model')

In [ ]:
# Difference distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models = ['Opus', 'Sonnet', 'Haiku']
colors = ['#3498db', '#9b59b6', '#e74c3c']

for i, (model, color) in enumerate(zip(models, colors)):
    data = scores_pivot[f'{model}_diff'].dropna()
    axes[i].hist(data, bins=20, color=color, alpha=0.7, edgecolor='black')
    axes[i].axvline(x=0, color='black', linestyle='--', linewidth=2)
    axes[i].axvline(x=data.mean(), color='red', linestyle='-', linewidth=2, label=f'Mean: {data.mean():.1f}')
    axes[i].set_title(f'{model} - Professor\nScore Differences', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Difference (AI - Professor)')
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

plt.tight_layout()
plt.savefig('score_differences.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Analysis by Project Type

In [ ]:
# Statistics by project type and evaluator
type_stats = df.groupby(['project_type', 'evaluator'])['vkupno'].agg(['mean', 'std', 'count']).round(2)
type_stats = type_stats.unstack(level=0)
type_stats

In [ ]:
# Grouped bar chart by project type
fig, ax = plt.subplots(figsize=(12, 6))

type_means = df.groupby(['project_type', 'evaluator'])['vkupno'].mean().unstack()
type_means = type_means[['Professor', 'Opus', 'Sonnet', 'Haiku']]

type_means.plot(kind='bar', ax=ax, width=0.8, color=['#2ecc71', '#3498db', '#9b59b6', '#e74c3c'])

ax.set_title('Average Scores by Project Type and Evaluator', fontsize=14, fontweight='bold')
ax.set_xlabel('Project Type')
ax.set_ylabel('Average Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Evaluator', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_ylim(0, 100)

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', fontsize=8, padding=2)

plt.tight_layout()
plt.savefig('scores_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Agreement Analysis

In [ ]:
# Define agreement thresholds
def categorize_agreement(diff, threshold=10):
    if abs(diff) <= threshold:
        return 'Close Agreement (±10)'
    elif diff > threshold:
        return 'AI Higher'
    else:
        return 'AI Lower'

# Categorize each model
agreement_stats = {}
for model in ['Opus', 'Sonnet', 'Haiku']:
    categories = scores_pivot[f'{model}_diff'].apply(categorize_agreement)
    counts = categories.value_counts()
    total = len(categories.dropna())
    agreement_stats[model] = {
        'Close Agreement': f"{counts.get('Close Agreement (±10)', 0)} ({counts.get('Close Agreement (±10)', 0)/total*100:.1f}%)",
        'AI Higher': f"{counts.get('AI Higher', 0)} ({counts.get('AI Higher', 0)/total*100:.1f}%)",
        'AI Lower': f"{counts.get('AI Lower', 0)} ({counts.get('AI Lower', 0)/total*100:.1f}%)"
    }

agreement_df = pd.DataFrame(agreement_stats).T
print("Agreement Categories (threshold = ±10 points):")
agreement_df

In [ ]:
# Stacked bar chart for agreement categories
fig, ax = plt.subplots(figsize=(10, 6))

agreement_data = []
for model in ['Opus', 'Sonnet', 'Haiku']:
    categories = scores_pivot[f'{model}_diff'].apply(categorize_agreement)
    counts = categories.value_counts(normalize=True) * 100
    agreement_data.append({
        'Model': model,
        'Close Agreement': counts.get('Close Agreement (±10)', 0),
        'AI Higher': counts.get('AI Higher', 0),
        'AI Lower': counts.get('AI Lower', 0)
    })

agree_df = pd.DataFrame(agreement_data).set_index('Model')
agree_df[['Close Agreement', 'AI Higher', 'AI Lower']].plot(
    kind='barh', stacked=True, ax=ax,
    color=['#27ae60', '#e74c3c', '#3498db']
)

ax.set_title('Agreement with Professor (±10 points threshold)', fontsize=14, fontweight='bold')
ax.set_xlabel('Percentage of Projects')
ax.set_ylabel('AI Model')
ax.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xlim(0, 100)

plt.tight_layout()
plt.savefig('agreement_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Which AI Model is Closest to the Professor?

In [ ]:
# Comprehensive comparison
comparison_metrics = pd.DataFrame({
    'Metric': [
        'Pearson Correlation (r)',
        'Mean Absolute Error (MAE)',
        'Root Mean Square Error (RMSE)',
        'Mean Bias (AI - Prof)',
        'Close Agreement Rate (±10 pts)',
        'Within ±5 pts Rate'
    ]
})

for model in ['Opus', 'Sonnet', 'Haiku']:
    valid = scores_pivot[['Professor', model]].dropna()
    diff = scores_pivot[f'{model}_diff'].dropna()
    
    r = stats.pearsonr(valid['Professor'], valid[model])[0]
    mae = diff.abs().mean()
    rmse = np.sqrt((diff**2).mean())
    bias = diff.mean()
    close_10 = (diff.abs() <= 10).mean() * 100
    close_5 = (diff.abs() <= 5).mean() * 100
    
    comparison_metrics[model] = [f'{r:.3f}', f'{mae:.2f}', f'{rmse:.2f}', f'{bias:+.2f}', f'{close_10:.1f}%', f'{close_5:.1f}%']

comparison_metrics.set_index('Metric')

In [ ]:
# Radar chart for model comparison
from math import pi

# Prepare data for radar (normalize to 0-1 scale, higher = better)
radar_data = {}
for model in ['Opus', 'Sonnet', 'Haiku']:
    valid = scores_pivot[['Professor', model]].dropna()
    diff = scores_pivot[f'{model}_diff'].dropna()
    
    r = stats.pearsonr(valid['Professor'], valid[model])[0]
    mae = diff.abs().mean()
    close_10 = (diff.abs() <= 10).mean()
    bias = abs(diff.mean())
    
    radar_data[model] = {
        'Correlation': r,
        'Low MAE': 1 - mae/50,  # Invert and normalize
        'Agreement Rate': close_10,
        'Low Bias': 1 - bias/30,  # Invert and normalize
        'Consistency': 1 - diff.std()/30  # Lower std = better
    }

# Create radar chart
categories = list(radar_data['Opus'].keys())
N = len(categories)

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

colors = ['#3498db', '#9b59b6', '#e74c3c']
for (model, values), color in zip(radar_data.items(), colors):
    values_list = list(values.values())
    values_list += values_list[:1]
    ax.plot(angles, values_list, 'o-', linewidth=2, label=model, color=color)
    ax.fill(angles, values_list, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('AI Model Performance Comparison\n(Higher = Better)', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig('model_comparison_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Individual Component Analysis

In [ ]:
# Compare individual scoring components
components = ['metap', 'tocnost', 'kviz']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, comp in enumerate(components):
    comp_means = df.groupby('evaluator')[comp].mean()
    comp_means = comp_means.reindex(['Professor', 'Opus', 'Sonnet', 'Haiku'])
    
    bars = axes[i].bar(comp_means.index, comp_means.values, 
                       color=['#2ecc71', '#3498db', '#9b59b6', '#e74c3c'])
    
    titles = {'metap': 'Metadata Factor (metap)', 'tocnost': 'Accuracy Factor (tocnost)', 'kviz': 'Quiz Score (kviz)'}
    axes[i].set_title(titles[comp], fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Average Value')
    
    # Add value labels
    for bar, val in zip(bars, comp_means.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                    f'{val:.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('component_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Extreme Cases Analysis

In [ ]:
# Find projects with largest disagreements
print("Projects with Largest Disagreements (AI vs Professor):")
print("="*70)

for model in ['Opus', 'Sonnet', 'Haiku']:
    diff_col = f'{model}_diff'
    
    # Largest positive difference (AI scored much higher)
    max_idx = scores_pivot[diff_col].idxmax()
    max_row = scores_pivot.loc[max_idx]
    
    # Largest negative difference (AI scored much lower)
    min_idx = scores_pivot[diff_col].idxmin()
    min_row = scores_pivot.loc[min_idx]
    
    print(f"\n{model}:")
    print(f"  Highest over-score: {max_row['project_id']} | Prof: {max_row['Professor']:.1f} | {model}: {max_row[model]:.1f} | Diff: +{max_row[diff_col]:.1f}")
    print(f"  Highest under-score: {min_row['project_id']} | Prof: {min_row['Professor']:.1f} | {model}: {min_row[model]:.1f} | Diff: {min_row[diff_col]:.1f}")

In [ ]:
# Heatmap of all scores
fig, ax = plt.subplots(figsize=(14, 10))

heatmap_data = scores_pivot.set_index('project_id')[['Professor', 'Opus', 'Sonnet', 'Haiku']]
heatmap_data = heatmap_data.dropna()

sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='RdYlGn', 
            center=50, vmin=0, vmax=100, ax=ax, cbar_kws={'label': 'Score'})

ax.set_title('All Evaluations Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('Evaluator')
ax.set_ylabel('Project ID')

plt.tight_layout()
plt.savefig('scores_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Key Findings Summary

In [ ]:
# Generate summary
print("="*70)
print("KEY FINDINGS SUMMARY")
print("="*70)

# Find best model
best_corr = max([(m, stats.pearsonr(scores_pivot['Professor'].dropna(), scores_pivot[m].dropna())[0]) 
                 for m in ['Opus', 'Sonnet', 'Haiku']], key=lambda x: x[1])
best_mae = min([(m, scores_pivot[f'{m}_diff'].abs().mean()) 
                for m in ['Opus', 'Sonnet', 'Haiku']], key=lambda x: x[1])

prof_mean = df[df['evaluator'] == 'Professor']['vkupno'].mean()

print(f"\n1. SAMPLE SIZE")
print(f"   - Total projects evaluated: {scores_pivot['project_id'].nunique()}")
print(f"   - Project types: {df['project_type'].unique().tolist()}")

print(f"\n2. OVERALL SCORING PATTERNS")
print(f"   - Professor average: {prof_mean:.1f}")
for model in ['Opus', 'Sonnet', 'Haiku']:
    ai_mean = df[df['evaluator'] == model]['vkupno'].mean()
    diff = ai_mean - prof_mean
    print(f"   - {model} average: {ai_mean:.1f} ({diff:+.1f} vs Professor)")

print(f"\n3. BEST PERFORMING AI MODEL")
print(f"   - Highest correlation with Professor: {best_corr[0]} (r = {best_corr[1]:.3f})")
print(f"   - Lowest Mean Absolute Error: {best_mae[0]} (MAE = {best_mae[1]:.2f})")

print(f"\n4. AGREEMENT RATES (within ±10 points)")
for model in ['Opus', 'Sonnet', 'Haiku']:
    diff = scores_pivot[f'{model}_diff'].dropna()
    rate = (diff.abs() <= 10).mean() * 100
    print(f"   - {model}: {rate:.1f}%")

print(f"\n5. SCORING BIAS")
for model in ['Opus', 'Sonnet', 'Haiku']:
    bias = scores_pivot[f'{model}_diff'].mean()
    direction = "higher" if bias > 0 else "lower"
    print(f"   - {model} tends to score {abs(bias):.1f} points {direction} than Professor")

print("\n" + "="*70)

In [ ]:
# Final recommendation
print("\nRECOMMENDATION:")
print("-" * 50)

# Score each model
model_scores = {}
for model in ['Opus', 'Sonnet', 'Haiku']:
    valid = scores_pivot[['Professor', model]].dropna()
    diff = scores_pivot[f'{model}_diff'].dropna()
    
    r = stats.pearsonr(valid['Professor'], valid[model])[0]
    mae = diff.abs().mean()
    bias = abs(diff.mean())
    
    # Combined score (weighted)
    score = r * 0.4 + (1 - mae/50) * 0.3 + (1 - bias/30) * 0.3
    model_scores[model] = score

best_model = max(model_scores, key=model_scores.get)
print(f"Based on correlation, MAE, and bias analysis:")
print(f"'{best_model}' shows the best overall alignment with Professor evaluations.")
print(f"\nModel scores (higher = better):")
for model, score in sorted(model_scores.items(), key=lambda x: -x[1]):
    print(f"  {model}: {score:.3f}")

---

## Export to HTML

Run this cell to export the notebook to a standalone HTML file:

In [ ]:
# This cell will export the notebook to HTML when run
import subprocess
import os

notebook_name = 'comparative_analysis.ipynb'
output_name = 'comparative_analysis_report.html'

# Check if we're in the right directory
if os.path.exists(notebook_name):
    result = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'html', '--no-input', notebook_name, '--output', output_name],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"Successfully exported to: {output_name}")
    else:
        print(f"Export failed: {result.stderr}")
        print("You can manually export using: jupyter nbconvert --to html comparative_analysis.ipynb")
else:
    print(f"Notebook not found. Save this notebook first, then re-run this cell.")